# Silver — Food Inspections (dev) — v2

Typed, validated, deduplicated layer over bronze:

* **Struct-typed schema** defined after peeking at the bronze data, enforced on every row with ANSI-safe `try_cast` (failed conversions become NULL)
* **Required fields**: `inspection_id, dba_name, license_, risk, inspection_date, results` — rows missing any of them (including failed casts) go to the quarantine table `workspace.foodguard_ai.food_inspections_quarantine`
* **Deduplication**: unique on `(inspection_id, inspection_date, results)`
* **`record_id`**: autogenerated `sha2` hash of `(inspection_id, inspection_date, results)` — unique across the entire silver layer
* **`is_active`**: SCD flag by ingestion timestamp — the newest version (`_ingested_at`) of each inspection `(inspection_id, inspection_date)` is active; superseded versions are kept with `is_active = false`

Output: one full snapshot per run at `silver/food_inspections/<ingest-date>/` (single part file, overwrite). Optional `PRUNE_OLD_SNAPSHOTS` flag (off by default) removes superseded snapshot folders so the whole silver volume stays unique on `record_id`. Raw history always remains in bronze.

In [0]:
# Source, destination, and catalog objects (bronze -> silver)
from datetime import date

INGEST_DATE = date.today().isoformat()
BRONZE_PATH = f"/Volumes/workspace/foodguard_ai/bronze/food_inspections/{INGEST_DATE}"
SILVER_BASE = "/Volumes/workspace/foodguard_ai/silver/food_inspections"
TARGET_PATH = f"{SILVER_BASE}/{INGEST_DATE}"
QUARANTINE_TABLE = "workspace.foodguard_ai.food_inspections_quarantine"

print(f"Bronze     : {BRONZE_PATH}")
print(f"Silver out : {TARGET_PATH}")
print(f"Quarantine : {QUARANTINE_TABLE}")

In [0]:
# Peek at the bronze data: structure + sample values (all STRING, plus audit columns).
bronze_df = spark.read.parquet(BRONZE_PATH)

bronze_df.printSchema()
print(f"Rows: {bronze_df.count()}")
display(bronze_df.limit(10))

In [0]:
# Silver schema as a StructType, informed by the peek above: numeric IDs and zip
# are longs, inspection_date is a timestamp; free-text and partially malformed
# columns (latitude / longitude / location, names) stay STRING.
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType,
)

SILVER_SCHEMA = StructType([
    StructField("inspection_id",   LongType(),      True),
    StructField("dba_name",        StringType(),    True),
    StructField("aka_name",        StringType(),    True),
    StructField("license_",        LongType(),      True),
    StructField("facility_type",   StringType(),    True),
    StructField("risk",            StringType(),    True),
    StructField("address",         StringType(),    True),
    StructField("city",            StringType(),    True),
    StructField("state",            StringType(),    True),
    StructField("zip",             LongType(),      True),
    StructField("inspection_date", TimestampType(), True),
    StructField("inspection_type", StringType(),    True),
    StructField("results",         StringType(),    True),
    StructField("violations",      StringType(),    True),
    StructField("latitude",        StringType(),    True),
    StructField("longitude",       StringType(),    True),
    StructField("location",        StringType(),    True),
    StructField("_source",         StringType(),    True),
    StructField("_ingested_at",    TimestampType(), True),  # ISO string -> timestamp, for SCD ordering
])

# Enforce the struct schema on ALL rows. try_cast is ANSI-safe: a failed
# conversion becomes NULL (a NULL in a required field quarantines the row).
silver_df = bronze_df.select([
    F.expr(f"try_cast(`{f.name}` as {f.dataType.simpleString()})").alias(f.name)
    for f in SILVER_SCHEMA.fields
])

silver_df.printSchema()

In [0]:
# Required fields: a row missing ANY of them (null in bronze, or null because the
# cast failed) is quarantined to the Unity Catalog table. Raw pre-cast values
# always remain in bronze for forensics.
REQUIRED = ["inspection_id", "dba_name", "license_", "risk", "inspection_date", "results"]
_all_not_null = " AND ".join(f"`{c}` IS NOT NULL" for c in REQUIRED)
_any_null = " OR ".join(f"`{c}` IS NULL" for c in REQUIRED)

valid_df = silver_df.where(_all_not_null)

quarantine_records = (
    silver_df.where(_any_null)
    .withColumn(
        "reason",
        F.concat_ws(", ", *[F.when(F.col(c).isNull(), F.lit(c)) for c in REQUIRED]),
    )
    .withColumn("quarantined_at", F.current_timestamp())
    .dropDuplicates(silver_df.columns)
)

# Create the quarantine table in the catalog (idempotent).
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} (
        {", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in SILVER_SCHEMA.fields)},
        `reason` STRING,
        `quarantined_at` TIMESTAMP
    )
""")

quarantined_count = quarantine_records.count()
if quarantined_count > 0:
    quarantine_records.write.mode("append").saveAsTable(QUARANTINE_TABLE)
    print(f"Quarantined to {QUARANTINE_TABLE}: {quarantined_count} rows "
          f"(re-running re-appends the same rows — quarantine is an audit trail)")
else:
    print("No rows missing required fields — nothing quarantined.")

print(f"Valid rows: {valid_df.count()}")

In [0]:
# record_id: deterministic sha2 hash of the dedup key (inspection_id,
# inspection_date, results). The dedup rule — unique (inspection_id,
# inspection_date, results) — is exactly uniqueness on record_id.
batch_df = valid_df.withColumn(
    "record_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("inspection_id").cast("string"),
            F.col("inspection_date").cast("string"),
            F.col("results"),
        ),
        256,
    ),
).dropDuplicates(["record_id"])

print(f"Batch rows after dedup: {batch_df.count()} (from {valid_df.count()} valid)")

In [0]:
# SCD logic on the ingestion timestamp (_ingested_at):
#   * an unchanged record re-ingested later just refreshes its _ingested_at
#   * a changed `results` yields a NEW record_id (a new version); the newest
#     version of each inspection (inspection_id, inspection_date) is active
from pyspark.sql.window import Window

# Load the latest existing silver snapshot (any date folder; today's own folder
# counts on re-runs). Pre-v2 snapshots without record_id trigger a rebuild.
existing_df = None
try:
    folders = sorted(p.name.strip("/") for p in dbutils.fs.ls(SILVER_BASE) if p.isDir())
except Exception:
    folders = []

if folders:
    latest_path = f"{SILVER_BASE}/{folders[-1]}"
    prev_df = spark.read.parquet(latest_path)
    if "record_id" in prev_df.columns and "is_active" in prev_df.columns:
        existing_df = prev_df
        print(f"Merging with previous snapshot: {latest_path} ({prev_df.count()} rows)")
    else:
        print(f"{latest_path} is a pre-v2 snapshot (no record_id) — rebuilding the silver layer from this batch.")

if existing_df is not None:
    merged = (
        existing_df.drop("is_active")
        .unionByName(batch_df)
        .withColumn("_max_ingested", F.max("_ingested_at").over(Window.partitionBy("record_id")))
        .where(F.col("_ingested_at") == F.col("_max_ingested"))
        .drop("_max_ingested")
        .dropDuplicates(["record_id"])
    )
else:
    merged = batch_df

scd_window = Window.partitionBy("inspection_id", "inspection_date").orderBy(
    F.col("_ingested_at").desc()
)
final_df = (
    merged.withColumn("_rn", F.row_number().over(scd_window))
    .withColumn("is_active", F.col("_rn") == 1)
    .drop("_rn")
)

(
    final_df.coalesce(1)
    .write
    .mode("overwrite")
    .parquet(TARGET_PATH)
)
print(f"Written to {TARGET_PATH}: {final_df.count()} rows "
      f"({final_df.where(F.col('is_active')).count()} active)")

# The snapshot folder is the full current state of the silver layer; optionally
# remove superseded date folders so record_id stays unique across the entire
# volume. OFF by default — set PRUNE_OLD_SNAPSHOTS = True to enable. History
# always remains in bronze.
PRUNE_OLD_SNAPSHOTS = False
if PRUNE_OLD_SNAPSHOTS:
    for folder in folders:
        if folder != INGEST_DATE:
            dbutils.fs.rm(f"{SILVER_BASE}/{folder}", True)
            print(f"Removed superseded snapshot folder: {folder}")
elif len(folders) > 1:
    print(f"PRUNE_OLD_SNAPSHOTS=False — kept {len(folders)} snapshot folders; set True to prune and keep record_id unique across the entire volume")

In [0]:
# Final sanity: row counts, record_id uniqueness across the ENTIRE silver layer,
# required-field completeness, schema, SCD summary, and quarantine total.
check_df = spark.read.parquet(TARGET_PATH)
layer_df = spark.read.option("recursiveFileLookup", "true").parquet(SILVER_BASE)

out_count, final_count = check_df.count(), final_df.count()
unique_batch = check_df.select("record_id").distinct().count() == out_count
unique_layer = layer_df.select("record_id").distinct().count() == layer_df.count()
active_count = check_df.where(F.col("is_active")).count()
required_violations = check_df.where(_any_null).count()
schema_ok = check_df.dtypes == final_df.dtypes  # types persisted (nullability differs after the Parquet round-trip)
quarantine_total = spark.table(QUARANTINE_TABLE).count()

print(f"Rows             — final: {final_count} | parquet: {out_count} | match: {final_count == out_count}")
print(f"record_id unique — batch: {unique_batch} | entire silver layer: {unique_layer}")
print(f"SCD              — active: {active_count} | inactive: {out_count - active_count}")
print(f"Required fields  — violations in silver: {required_violations}")
print(f"Schema persisted : {schema_ok}")
print(f"Quarantine table rows (all runs): {quarantine_total}")
assert final_count == out_count, "Row count mismatch!"
assert unique_batch, "record_id not unique in batch!"
assert unique_layer, "record_id not unique across the silver layer!"
assert required_violations == 0, "Required fields missing in silver!"
assert schema_ok, "Schema mismatch — enforced types were not persisted!"
display(check_df.limit(5))